# 06 — Discrete-Time Hazard Analysis

**Author:** Ricardo Sanchez  

**Project:** Monetary Policy Shocks and Startup Funding Transitions (Seed → Series A)

**Date:** March 03, 2026 

**Data Source:** Person-period panel from `04_startup_panel_with_shocks_final.ipynb`

---

## Objective

Estimate discrete-time hazard models to test whether monetary policy shocks influence the timing of startup transitions from Seed to Series A funding. The complementary log-log (cloglog) link is used because the discrete quarterly intervals approximate an underlying continuous-time process.

## Motivation from Descriptive Analysis (Notebook 05)

The Kaplan-Meier analysis and log-rank tests revealed the following patterns that directly inform the model specifications below:

| Finding | p-value | Implication |
|---------|---------|-------------|
| FinTech vs. AI survival curves are **indistinguishable** | 0.962 | Sector FE included as control but not expected to be significant |
| SFBA transitions to Series A **significantly faster** than all other metros | 0.001–0.028 | Metro FE well-motivated; SFBA is the key contrast |
| Cohort-period survival curves **do not differ** significantly | 0.43–0.88 | Cohort FE retained as controls for unobserved time trends, but not driving the result |
| Startups seeded during **contractionary** shock quarters transition **significantly slower** | 0.008 vs. Neutral | Core finding — contractionary monetary policy delays VC deployment |
| Expansionary vs. Contractionary is also significant | 0.046 | Shock direction matters, not just magnitude |

## Three Model Specifications

| Model | Description | Rationale |
|-------|-------------|-----------|
| **Model 1 — Baseline** | Full specification: shocks + 5 lags, macro controls, firm covariates, duration/cohort/sector/metro FE | Establishes the core relationship with all standard controls |
| **Model 2 — Parsimonious** | Drops sector FE (p=0.96 in KM) and cohort FE (p>0.43 in KM); retains shocks, macro, firm covariates, duration, metro FE | Tests whether the shock effect survives in a leaner model without non-significant fixed effects |
| **Model 3 — SFBA Interaction** | Adds SFBA × shock interaction to the baseline | Tests whether the SF Bay Area's deep VC ecosystem buffers startups from contractionary shocks, or whether the effect is uniform across metros |

## Inputs
- `data/cleaned/discrete_time_hazard_data/startup_seriesA_person_period_quarterly_final_all.csv`

## Outputs
- Model summary tables (inline)
- Coefficient comparison table

---

## Part 1 — Setup

### 1.1 Imports and Data Load

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [3]:
# Load the person-person panel (all feature engineering already applied in NB04)
PATH = r"..\data\cleaned\discrete_time_hazard_data\startup_seriesA_person_period_quarterly_final_all.csv"
df = pd.read_csv(PATH, parse_dates=["period_start","seed_date"])

In [7]:
# Quick data inventory
print(f"Panel dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique firms:     {df['firm_id'].nunique():,}")
print(f"Period range:     {df['period_start'].min():%Y-Q}{(df['period_start'].min().month-1)//3+1} to "
      f"{df['period_start'].max():%Y-Q}{(df['period_start'].max().month-1)//3+1}")
print(f"Total events:     {df['event_seriesA'].sum():,}")

Panel dimensions: 23,214 rows × 29 columns
Unique firms:     1,904
Period range:     2016-Q1 to 2023-Q3
Total events:     694


### 1.2 Construct Fixed-Effect Dummies

Fixed effects are created here because different model specifications use different subsets. The descriptive analysis (NB05) found that sector and cohort are not statistically significant in the KM curves, while metro region (especially SFBA) is highly significant.

In [8]:
# Cohort fixed effects by seed year (base = earliest year in sample)
# NB05 finding: cohort groups not significant (p > 0.43), but retained as controls
df["seed_year"] = df["seed_date"].dt.year
cohort_d = pd.get_dummies(df["seed_year"].astype("Int64"), prefix="cohortY", drop_first=True)

In [9]:
# Duration dummies from pre-computed bins (base = 0–2 quarters)
dur_d = pd.get_dummies(df["dur_bin"], drop_first=True)

In [10]:
# Sector dummies
# NB05 finding: FinTech vs AI survival curves are indistinguishable (p = 0.962)
# Included as control but not expected to reach significance
sector_d = pd.get_dummies(df["sector"].astype(str), prefix="sector", drop_first=True) if "sector" in df.columns else pd.DataFrame()

In [11]:
# Metro-region dummies
# NB05 finding: SFBA transitions significantly faster than all other metros (p = 0.001–0.028)
metro_d = pd.get_dummies(df["metro_region"].astype(str), prefix="metro", drop_first=True) if "metro_region" in df.columns else pd.DataFrame()

In [12]:
# SFBA indicator for Model 3 interaction 
df["is_sfba"] = (df["metro_region"] == "SFBA").astype(int)

In [13]:
# Clustering variable — standard errors clustered by calendar quarter
cluster_groups = df["period_start"].dt.to_period("Q").astype(str)

In [14]:
# Dependent variable — Series A event indicator
y = df["event_seriesA"].astype(int)

### 1.3 Pre-Model Cleaning & Estimation Helper

This function handles the design matrix diagnostics (dropping constant columns, detecting complete separation, checking numerical rank) and fits the cloglog GLM with quarter-clustered standard errors. It is called once per model specification to avoid duplicating this logic.

In [15]:
def fit_cloglog(X, y, groups, model_label="Model"):
    """
    Clean the design matrix and fit a discrete-time hazard model.
    
    Steps:
        0. Convert bool dummies to numeric
        1. Drop zero-variance / constant columns
        2. Drop exact duplicate columns
        3. Drop dummies with complete separation
        4. SVD rank check — reduce to full rank if needed
        5. Add constant and fit cloglog GLM with clustered SEs
    
    Returns the fitted GLM results object.
    """
    import numpy as np
    import statsmodels.api as sm
    
    X_clean = X.copy()
    y_clean = y.copy()
    
    # --- 0) Make sure types are numeric ---
    bool_cols = X_clean.select_dtypes(include=["bool"]).columns
    X_clean[bool_cols] = X_clean[bool_cols].astype(np.uint8)
    X_clean = X_clean.astype(float)
    y_clean = y_clean.astype(int)

    # --- 1) Drop zero-variance / constant columns ---
    const_cols = X_clean.columns[X_clean.nunique() <= 1].tolist()
    if const_cols:
        print(f"[{model_label}] Dropping constant cols: {const_cols}")
        X_clean = X_clean.drop(columns=const_cols)

    # --- 2) Drop exact duplicate columns ---
    dup_mask = X_clean.T.duplicated()
    if dup_mask.any():
        dups = X_clean.columns[dup_mask].tolist()
        print(f"[{model_label}] Dropping duplicate cols: {dups}")
        X_clean = X_clean.loc[:, ~dup_mask]

    # --- 3) Drop dummies that perfectly separate the outcome (complete separation) ---
    sep_cols = []
    for c in X_clean.columns:
        vals = X_clean[c].dropna().unique()
        if set(np.round(vals, 6)).issubset({0.0, 1.0}):
            s = X_clean[c] == 1
            if s.any() and (~s).any():
                y1 = y_clean[s]
                y0 = y_clean[~s]
                if (y1.nunique() == 1 and y0.nunique() == 1 and y1.iloc[0] != y0.iloc[0]) or \
                   (y1.nunique() == 1 and (y1.iloc[0] in [0,1]) and (y_clean[s].mean() in [0,1])):
                    sep_cols.append(c)
            elif s.any() and (y_clean[s].nunique() == 1):
                sep_cols.append(c)
    if sep_cols:
        print(f"[{model_label}] Dropping separation cols: {sep_cols}")
        X_clean = X_clean.drop(columns=sep_cols)

    # --- 4) SVD rank check ---
    U, svals, Vt = np.linalg.svd(X_clean.values, full_matrices=False)
    rank = (svals > 1e-10).sum()
    if rank < X_clean.shape[1]:
        keep_idx = np.argsort(-svals)[:rank]
        col_score = (Vt[:rank, :]**2).sum(axis=0)
        order = np.argsort(-col_score)[:rank]
        print(f"[{model_label}] Reducing to full rank: {X_clean.shape[1]} → {rank} columns")
        X_clean = X_clean.iloc[:, order]

    # --- 5) Add constant and fit ---
    X_clean = sm.add_constant(X_clean, has_constant="add")
    assert len(groups) == len(y_clean)

    model = sm.GLM(y_clean, X_clean,
                   family=sm.families.Binomial(link=sm.families.links.cloglog()))
    res = model.fit(cov_type="cluster", cov_kwds={"groups": groups})
    
    print(f"\n{'='*80}")
    print(f"{model_label}")
    print(f"{'='*80}")
    print(res.summary())
    
    return res

---

## Part 2 — Model 1: Baseline Specification

The full model with all controls. This establishes the core shock–hazard relationship while absorbing variation from duration dependence, cohort timing, sector, and geography.

**Regressors:**
- Monetary policy shock (contemporaneous) + 5 quarterly lags
- Macro controls: Core CPI (YoY %), Real GDP (YoY %)
- Firm covariates: log seed amount, log partner investors
- Fixed effects: duration bins, cohort year, sector, metro region

In [16]:
# Define core regressors for Model 1
core_m1 = [
    "shock_25bp_units", "shock25_lag1", "shock25_lag2", "shock25_lag3", "shock25_lag4", "shock25_lag5",
    "core_cpi_yoy", "real_gdp_yoy",
    "log_seed_amt", "log_npi"
]

# Assemble the full design matrix: core + duration FE + cohort FE + sector FE + metro FE
fe_list_m1 = [dur_d, cohort_d]
if len(sector_d): fe_list_m1.append(sector_d)
if len(metro_d):  fe_list_m1.append(metro_d)

X_m1 = pd.concat([df[core_m1]] + fe_list_m1, axis=1)

In [17]:
# Drop rows with any missing values in the design matrix
data_m1 = pd.concat([y, X_m1], axis=1).dropna()

y_m1 = data_m1["event_seriesA"].astype(int)
X_m1_clean = data_m1.drop(columns=["event_seriesA"])
groups_m1 = cluster_groups.loc[data_m1.index]

print(f"Model 1 sample: {len(data_m1):,} person-quarters  |  {y_m1.sum():,} events  |  {X_m1_clean.shape[1]} regressors")

Model 1 sample: 14,520 person-quarters  |  457 events  |  25 regressors


In [18]:
# Fit Model 1 — Baseline
res_m1 = fit_cloglog(X_m1_clean, y_m1, groups_m1, model_label="Model 1 — Baseline")

[Model 1 — Baseline] Dropping separation cols: ['cohortY_2023']


C:\Users\Ricardo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The cloglog link alias is deprecated. Use CLogLog instead. The cloglog link alias will be removed after the 0.15.0 release.
  warnings.warn(



Model 1 — Baseline
                 Generalized Linear Model Regression Results                  
Dep. Variable:          event_seriesA   No. Observations:                14520
Model:                            GLM   Df Residuals:                    14495
Model Family:                Binomial   Df Model:                           24
Link Function:                cloglog   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1847.9
Date:                Wed, 11 Mar 2026   Deviance:                       3695.8
Time:                        14:16:59   Pearson chi2:                 1.49e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.02481
Covariance Type:              cluster                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const              -

---

## Part 3 — Model 2: Parsimonious Specification

The NB05 descriptive analysis showed that neither sector (p = 0.962) nor cohort period (p > 0.43) significantly differentiates survival curves. This model drops both sector and cohort fixed effects to test whether the monetary policy shock effect is robust in a leaner specification.

**What changes vs. Model 1:**
- Sector FE removed (FinTech vs. AI survival curves are indistinguishable)
- Cohort year FE removed (no significant cohort-period differences)
- All other regressors retained

**What this tests:** If the shock coefficients remain stable after dropping these non-significant controls, it strengthens the claim that the finding is not an artifact of over-specification.

In [19]:
# ── Design matrix for Model 2: drop sector and cohort FE ──
fe_list_m2 = [dur_d]
if len(metro_d): fe_list_m2.append(metro_d)

X_m2 = pd.concat([df[core_m1]] + fe_list_m2, axis=1)

In [20]:
# Drop rows with any missing values
data_m2 = pd.concat([y, X_m2], axis=1).dropna()

y_m2 = data_m2["event_seriesA"].astype(int)
X_m2_clean = data_m2.drop(columns=["event_seriesA"])
groups_m2 = cluster_groups.loc[data_m2.index]

print(f"Model 2 sample: {len(data_m2):,} person-quarters  |  {y_m2.sum():,} events  |  {X_m2_clean.shape[1]} regressors")

Model 2 sample: 14,520 person-quarters  |  457 events  |  17 regressors


In [22]:
# Fit Model 2 — Parsimonious (drop sector and cohort FE)
res_m2 = fit_cloglog(X_m2_clean, y_m2, groups_m2, model_label="Model 2 — Parsimonious")


Model 2 — Parsimonious
                 Generalized Linear Model Regression Results                  
Dep. Variable:          event_seriesA   No. Observations:                14520
Model:                            GLM   Df Residuals:                    14502
Model Family:                Binomial   Df Model:                           17
Link Function:                cloglog   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1868.5
Date:                Wed, 11 Mar 2026   Deviance:                       3737.0
Time:                        14:20:14   Pearson chi2:                 1.46e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.02204
Covariance Type:              cluster                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const           

C:\Users\Ricardo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The cloglog link alias is deprecated. Use CLogLog instead. The cloglog link alias will be removed after the 0.15.0 release.
  warnings.warn(


---

## Part 4 — Model 3: SFBA × Shock Interaction

The NB05 metro analysis showed that the SF Bay Area has a **significantly faster** Seed-to-Series A transition than every other metro region (p = 0.001 vs. GLA, p = 0.028 vs. GMA, p = 0.027 vs. GNYA). This raises a natural question: does SFBA's deep venture capital ecosystem **buffer** startups from the negative effects of contractionary monetary policy shocks, or is the shock effect uniform across geographies?

**What changes vs. Model 1:**
- Adds `is_sfba × shock_25bp_units` interaction term
- Adds `is_sfba × shock25_lag1` interaction term (to capture delayed buffering)
- All baseline controls retained

**How to interpret:**
- If the interaction terms are **negative**, SFBA *amplifies* the shock effect
- If the interaction terms are **positive**, SFBA *buffers* against contractionary shocks
- If the interaction terms are near zero / insignificant, the shock effect is **geographically uniform**

In [23]:
# Build interaction terms: SFBA × contemporaneous shock and SFBA × lag 1
df["sfba_x_shock"] = df["is_sfba"] * df["shock_25bp_units"]
df["sfba_x_shock_lag1"] = df["is_sfba"] * df["shock25_lag1"]

# Core regressors for Model 3: baseline + interaction terms
core_m3 = core_m1 + ["sfba_x_shock", "sfba_x_shock_lag1"]

# Full design matrix with all FE (same as Model 1)
X_m3 = pd.concat([df[core_m3]] + fe_list_m1, axis=1)

In [24]:
# Drop rows with any missing values
data_m3 = pd.concat([y, X_m3], axis=1).dropna()

y_m3 = data_m3["event_seriesA"].astype(int)
X_m3_clean = data_m3.drop(columns=["event_seriesA"])
groups_m3 = cluster_groups.loc[data_m3.index]

print(f"Model 3 sample: {len(data_m3):,} person-quarters  |  {y_m3.sum():,} events  |  {X_m3_clean.shape[1]} regressors")

Model 3 sample: 14,520 person-quarters  |  457 events  |  27 regressors


In [25]:
# Fit Model 3 — SFBA Interaction
res_m3 = fit_cloglog(X_m3_clean, y_m3, groups_m3, model_label="Model 3 — SFBA × Shock Interaction")

[Model 3 — SFBA × Shock Interaction] Dropping separation cols: ['cohortY_2023']


C:\Users\Ricardo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\genmod\families\links.py:13: FutureWarning: The cloglog link alias is deprecated. Use CLogLog instead. The cloglog link alias will be removed after the 0.15.0 release.
  warnings.warn(



Model 3 — SFBA × Shock Interaction
                 Generalized Linear Model Regression Results                  
Dep. Variable:          event_seriesA   No. Observations:                14520
Model:                            GLM   Df Residuals:                    14493
Model Family:                Binomial   Df Model:                           26
Link Function:                cloglog   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -1847.0
Date:                Wed, 11 Mar 2026   Deviance:                       3694.0
Time:                        14:22:21   Pearson chi2:                 1.49e+04
No. Iterations:                     8   Pseudo R-squ. (CS):            0.02493
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
co

---

## Part 5 — Model Comparison

Compare the three specifications side-by-side on key fit statistics and shock variable coefficients. Stability of the shock coefficients across specifications strengthens the causal interpretation.

In [26]:
# Fit statistics comparison
comparison = pd.DataFrame({
    "Model 1 (Baseline)":      {"AIC": res_m1.aic, "BIC": res_m1.bic, "Log-Lik": res_m1.llf, "N obs": int(res_m1.nobs), "Regressors": res_m1.df_model},
    "Model 2 (Parsimonious)":  {"AIC": res_m2.aic, "BIC": res_m2.bic, "Log-Lik": res_m2.llf, "N obs": int(res_m2.nobs), "Regressors": res_m2.df_model},
    "Model 3 (SFBA Interact)": {"AIC": res_m3.aic, "BIC": res_m3.bic, "Log-Lik": res_m3.llf, "N obs": int(res_m3.nobs), "Regressors": res_m3.df_model},
}).T

print("── Model Fit Comparison ──")
print(comparison.to_string())

── Model Fit Comparison ──
                                 AIC            BIC      Log-Lik    N obs  Regressors
Model 1 (Baseline)       3745.836642 -135213.840128 -1847.918321  14520.0        24.0
Model 2 (Parsimonious)   3772.993252 -135239.766494 -1868.496626  14520.0        17.0
Model 3 (SFBA Interact)  3748.022805 -135196.487400 -1847.011403  14520.0        26.0


C:\Users\Ricardo\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1923: FutureWarning: The bic value is computed using the deviance formula. After 0.13 this will change to the log-likelihood based formula. This change has no impact on the relative rank of models compared using BIC. You can directly access the log-likelihood version using the `bic_llf` attribute. You can suppress this message by calling statsmodels.genmod.generalized_linear_model.SET_USE_BIC_LLF with True to get the LLF-based version now or False to retainthe deviance version.
  warnings.warn(


In [30]:
# Shock coefficient comparison across models 
shock_vars = ["shock_25bp_units", "shock25_lag1", "shock25_lag2", "shock25_lag3", "shock25_lag4", "shock25_lag5"]

rows = []
for var in shock_vars:
    row = {"Variable": var}
    for label, res in [("M1 Baseline", res_m1), ("M2 Parsimonious", res_m2), ("M3 SFBA Interact", res_m3)]:
        if var in res.params.index:
            coef = res.params[var]
            pval = res.pvalues[var]
            sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
            row[f"{label} coef"] = f"{coef:+.4f}{sig}"
            row[f"{label} p"] = f"{pval:.3f}"
        else:
            row[f"{label} coef"] = "—"
            row[f"{label} p"] = "—"
    rows.append(row)

# Add interaction terms from Model 3
for var in ["sfba_x_shock", "sfba_x_shock_lag1"]:
    row = {"Variable": var}
    for label, res in [("M1 Baseline", res_m1), ("M2 Parsimonious", res_m2), ("M3 SFBA Interact", res_m3)]:
        if var in res.params.index:
            coef = res.params[var]
            pval = res.pvalues[var]
            sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
            row[f"{label} coef"] = f"{coef:+.4f}{sig}"
            row[f"{label} p"] = f"{pval:.3f}"
        else:
            row[f"{label} coef"] = "—"
            row[f"{label} p"] = "—"
    rows.append(row)

shock_comp = pd.DataFrame(rows).set_index("Variable")
print("\n── Shock Coefficient Comparison Across Models ──")
print(shock_comp.to_string())


── Shock Coefficient Comparison Across Models ──
                  M1 Baseline coef M1 Baseline p M2 Parsimonious coef M2 Parsimonious p M3 SFBA Interact coef M3 SFBA Interact p
Variable                                                                                                                        
shock_25bp_units           -0.0267         0.803              -0.0375             0.777               -0.1758              0.105
shock25_lag1               -0.1771         0.422              -0.1673             0.484               -0.0303              0.896
shock25_lag2               -0.2396         0.202              -0.1289             0.493               -0.2437              0.193
shock25_lag3             -0.3514**         0.014            -0.3020**             0.028             -0.3556**              0.011
shock25_lag4              -0.2906*         0.074              -0.2464             0.215              -0.2922*              0.069
shock25_lag5               -0.0026         0.98

In [31]:
# Key control variables comparison
control_vars = ["core_cpi_yoy", "real_gdp_yoy", "log_seed_amt", "log_npi"]

rows = []
for var in control_vars:
    row = {"Variable": var}
    for label, res in [("M1 Baseline", res_m1), ("M2 Parsimonious", res_m2), ("M3 SFBA Interact", res_m3)]:
        if var in res.params.index:
            coef = res.params[var]
            pval = res.pvalues[var]
            sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
            row[f"{label} coef"] = f"{coef:+.4f}{sig}"
            row[f"{label} p"] = f"{pval:.3f}"
        else:
            row[f"{label} coef"] = "—"
            row[f"{label} p"] = "—"
    rows.append(row)

ctrl_comp = pd.DataFrame(rows).set_index("Variable")
print("\n── Control Variable Comparison Across Models ──")
print(ctrl_comp.to_string())


── Control Variable Comparison Across Models ──
             M1 Baseline coef M1 Baseline p M2 Parsimonious coef M2 Parsimonious p M3 SFBA Interact coef M3 SFBA Interact p
Variable                                                                                                                   
core_cpi_yoy          +0.0811         0.307              -0.0169             0.834               +0.0812              0.304
real_gdp_yoy       +0.0790***         0.000           +0.0846***             0.000            +0.0793***              0.000
log_seed_amt       +0.4397***         0.000           +0.4029***             0.000            +0.4393***              0.000
log_npi            +0.4745***         0.000           +0.4878***             0.000            +0.4751***              0.000


---

## Part 6 — Interpretation Notes

**Model 1 (Baseline)** provides the benchmark estimate of the shock–hazard relationship with all standard controls in place. This is the specification to report as the primary result.

**Model 2 (Parsimonious)** tests robustness. If the shock coefficients are stable when sector and cohort FE are dropped (both were non-significant in the KM analysis), it confirms the result is not sensitive to including non-informative controls. A large change in AIC/BIC between Models 1 and 2 would indicate whether those FE were contributing noise or absorbing meaningful variation.

**Model 3 (SFBA Interaction)** addresses the geographic heterogeneity story. The SFBA metro has a significantly faster transition rate, and the interaction terms test whether that advantage extends to shock resilience. A positive, significant `sfba_x_shock` coefficient would mean SFBA startups are partially insulated from contractionary shocks — consistent with the theory that deeper, more liquid VC markets smooth funding through tight-money periods.